# FinBERT News Sentiment (run on Colab's free GPU/RAM)

This notebook fetches recent news headlines for a company, scores each one with the real FinBERT model (not the lightweight VADER fallback used locally), and produces a CSV you can download and hand back to the local Streamlit project.

**How to use:** Runtime menu -> Run all. When it finishes, `finbert_sentiment.csv` will download automatically to your computer's Downloads folder.

In [ ]:
!pip install -q feedparser --quiet

In [ ]:
import urllib.parse
import urllib.request
from datetime import datetime, timedelta

import feedparser
import pandas as pd


def fetch_headlines(query: str, period_days: int) -> pd.DataFrame:
    """Uses Google News' public RSS feed. Note: it returns roughly its ~100 most
    recent results for the query, so it's recency-biased regardless of period_days
    (we found this out the hard way running this same code locally)."""
    encoded_query = urllib.parse.quote(query)
    url = f"https://news.google.com/rss/search?q={encoded_query}&hl=en-US&gl=US&ceid=US:en"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    data = urllib.request.urlopen(req, timeout=15).read()
    feed = feedparser.parse(data)

    cutoff = datetime.utcnow() - timedelta(days=period_days)
    rows = []
    for entry in feed.entries:
        if not getattr(entry, "published_parsed", None):
            continue
        published = datetime(*entry.published_parsed[:6])
        if published >= cutoff:
            rows.append({"date": published.date(), "title": entry.title})
    return pd.DataFrame(rows)


QUERY = "Google"  # try the company name, often matches better than the ticker
PERIOD_DAYS = 60

headlines = fetch_headlines(QUERY, PERIOD_DAYS)
print(f"Fetched {len(headlines)} headlines, {headlines['date'].nunique()} unique dates")
headlines.head()

In [ ]:
# Loaded via the specific Bert classes (not AutoConfig/AutoModel) because
# finbert-tone's config.json predates the model_type field that Auto* dispatch
# requires on newer transformers releases. This is also the loading method the
# model's own Hugging Face page documents.
from transformers import BertTokenizer, BertForSequenceClassification, pipeline

tokenizer = BertTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model = BertForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone", num_labels=3)
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

In [ ]:
def signed_score(title: str) -> float:
    result = classifier(title[:512])[0]
    sign = {"Positive": 1, "Negative": -1, "Neutral": 0}.get(result["label"], 0)
    return sign * result["score"]


headlines["sentiment"] = headlines["title"].apply(signed_score)
headlines.sort_values("date")

In [ ]:
OUTPUT_FILE = "finbert_sentiment.csv"
headlines.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(headlines)} scored headlines to {OUTPUT_FILE}")

try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    print("Not running on Colab — find the file in the current directory instead.")